In [9]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import os
import torch
from torchvision import transforms
import onnxruntime 
import time
import openvino as ov
# import ipywidgets as widgets
import sys


In [ ]:
sys.path.insert(0,"../")
def post_process(input_person,input_clothes):
    #由于没有对数据预处理，当直接输入图片时，导致无法得到理想结果，因此对数据进行预处理，预处理的方法需要自己的源码中所使用的方法。
    #输入预处理
    load_width = 192

    transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
            ])
    input_clothes = transforms.Resize(load_width, interpolation=2)(input_clothes)
    input_person = transforms.Resize(load_width, interpolation=2)(input_person)
    input_clothes = transform(input_clothes).unsqueeze(0)  #torch.Size([1,3, 256, 192])
    input_person = transform(input_person).unsqueeze(0)#torch.Size([1,3, 256, 192])

    input_clothes_np = np.array(input_clothes).astype(np.float32)
    input_person_np = np.array(input_person).astype(np.float32)
    return input_person,input_clothes

def orimodel(model,input_person,input_clothes):
    #对原模型的时间进行统计
    start_time = time.time()
    ori_output = model(input_person,input_clothes)
    end_time = time.time()
    # 计算并打印运行时间
    execution_time = end_time - start_time
    print(f"Execution time: {execution_time:.4f} seconds")

    #对原模型输出处理
    ori_output_np = np.array(ori_output.detach())
    value_range = (np.min(ori_output_np),np.max(ori_output_np)) #此处进行归一化
    ori_output_result = ((ori_output_np - value_range[0]) / (value_range[1] - value_range[0])) * 255
    ori_output_result = ori_output_result.astype(np.uint8)  # 将数据转换为 uint8

    ori_output_result = np.transpose(ori_output_result, (2, 3, 1, 0))

    ori_output_result = np.squeeze(ori_output_result)
    return ori_output_result

In [ ]:
# 原模型


model_path='/mnt/bb/yc/virtual-tryon-ds/checkpoint/checkpoint_bypy/pf_b_2l2e-05_0.5_/pf_b_2l2e-05_0.5__data.pt'
# model_path='/mnt/bb/yc/virtual-tryon-ds/checkpoint/checkpoint_bypy/pf_b_2l2e-05m_35/pf_b_2l2e-05m_35_data.pt'
# model_path='/mnt/bb/yc/virtual-tryon-ds/checkpoint/checkpoint_bypy/pf_b_2l2e-05m_63/pf_b_2l2e-05m_63_data.pt'
onnx_path=model_path.split('.')[0]+'.onnx'
device = "cpu"
input_clothes_path='/mnt/bb/yc/virtual-tryon-ds/data/VITON/VITON/VITON_test/clothes/000010_1.jpg'
input_person_path="/mnt/bb/yc/virtual-tryon-ds/data/VITON/VITON/VITON_test/test_img/000228_0.jpg"
input_clothes = Image.open(input_clothes_path).convert('RGB')
input_person = Image.open(input_person_path).convert('RGB')
plt_clothes = input_clothes #拷贝一份，用于之后绘制图像
plt_person = input_person

model = torch.load(model_path,map_location='cpu')
model = model.to(device)
input_person,input_clothes = post_process(input_person,input_clothes)


In [ ]:
import os
from PIL import Image

# 定义路径
txt_path = '/mnt/bb/yc/virtual-tryon-ds/data/VITON/test_any.txt'  # 替换为你的txt文件路径
person_dir = '/mnt/bb/yc/virtual-tryon-ds/data/VITON/VITON/VITON_test/test_img/'  # 人物图片目录
clothes_dir = '/mnt/bb/yc/virtual-tryon-ds/data/VITON/VITON/VITON_test/clothes/'  # 服装图片目录
output_dir = './output_images/'  # 输出目录

# 创建输出目录
os.makedirs(output_dir, exist_ok=True)

# 加载模型
model = torch.load(model_path, map_location='cpu')
model = model.to(device)

# 读取txt文件并处理每一行
with open(txt_path, 'r') as f:
    for line in f:
        # 分割每行得到两个文件名
        person_img, clothes_img = line.strip().split()
        
        # 构建完整文件路径
        person_path = os.path.join(person_dir, person_img)
        clothes_path = os.path.join(clothes_dir, clothes_img)
        
        # 检查文件是否存在
        if not os.path.exists(person_path):
            print(f"人物图片不存在: {person_path}")
            continue
        if not os.path.exists(clothes_path):
            print(f"服装图片不存在: {clothes_path}")
            continue
            
        # 打开图片
        input_person = Image.open(person_path).convert('RGB')
        input_clothes = Image.open(clothes_path).convert('RGB')
        
        # 预处理
        input_person_tensor, input_clothes_tensor = post_process(input_person, input_clothes)
        
        # 模型推理
        output_result = orimodel(model, input_person_tensor, input_clothes_tensor)
        
        # 保存结果
        output_img = Image.fromarray(output_result)
        output_filename = f"{os.path.splitext(person_img)[0]}_{os.path.splitext(clothes_img)[0]}.jpg"
        output_path = os.path.join(output_dir, output_filename)
        output_img.save(output_path)
        print(f"已保存: {output_path}")

print("所有图片处理完成！")

/mnt/zz/anaconda3/envs/torch2.0/lib/python3.8/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/mnt/zz/anaconda3/envs/torch2.0/lib/python3.8/site-packages/torch/nn/functional.py:4316: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  warnings.warn(


Execution time: 0.9082 seconds
已保存: ./output_images/000786_0_007957_1.jpg
Execution time: 0.7648 seconds
已保存: ./output_images/000795_0_006859_1.jpg
Execution time: 0.7576 seconds
已保存: ./output_images/000803_0_005203_1.jpg
Execution time: 0.7798 seconds
已保存: ./output_images/000812_0_003821_1.jpg
Execution time: 0.7678 seconds
已保存: ./output_images/000827_0_002297_1.jpg
Execution time: 1.0662 seconds
已保存: ./output_images/000837_0_004710_1.jpg
Execution time: 0.7804 seconds
已保存: ./output_images/000846_0_001606_1.jpg
Execution time: 0.7566 seconds
已保存: ./output_images/000855_0_011816_1.jpg
Execution time: 0.7970 seconds
已保存: ./output_images/000867_0_002111_1.jpg
Execution time: 0.8137 seconds
已保存: ./output_images/000877_0_010543_1.jpg
Execution time: 0.9599 seconds
已保存: ./output_images/000889_0_004080_1.jpg
Execution time: 0.8186 seconds
已保存: ./output_images/000898_0_011173_1.jpg
Execution time: 0.8340 seconds
已保存: ./output_images/000907_0_004123_1.jpg
Execution time: 0.8170 seconds
已保存: ./